# Phase Voice: Embed Qwen2.5-VL-3B into Flux-Apex

**Transform FLUX into a self-contained multimodal AI system**

This notebook:
1. Downloads Flux-Apex-V1.flx from HuggingFace
2. Downloads Qwen2.5-VL-3B-Instruct (text + vision)
3. Embeds the VL model directly into .flx
4. **Removes** the byte decoder (not legacy — completely strips it)
5. Updates to version 5.0-vlm-embedded
6. Saves back to Flux-Apex-V1.flx

**Why Qwen2.5-VL-3B?**
- Native transformers support (no trust_remote_code issues)
- Text + Vision capabilities
- Only ~3B params (~1.5GB quantized)
- Same Qwen family as original external LLM

**Expected sizes:**
- Current Flux-Apex-V1.flx: ~5.79 GB
- Decoder removed: -124 MB
- Qwen2.5-VL-3B (fp16) added: +6 GB
- **Final size: ~11.5 GB**

---

## Cell 1: Clone/Pull FLUX Repository

In [ ]:
%%time
import os
from pathlib import Path

REPO_URL = 'https://github.com/Unseengap/FLUX.git'
ROOT = Path('/kaggle/working/FLUX') if Path('/kaggle').exists() else Path('/content/FLUX')

# For local development
if not Path('/kaggle').exists() and not Path('/content').exists():
    ROOT = Path.cwd()
    if (ROOT / 'flux_utils.py').exists():
        print(f'  Using local directory: {ROOT}')
    else:
        ROOT = Path('/Users/admin/Desktop/flux')

if ROOT.exists() and (ROOT / '.git').exists():
    print('  Pulling latest...')
    os.chdir(ROOT)
    !git pull --ff-only 2>/dev/null || echo '  (pull skipped)'
elif not ROOT.exists():
    print('  Cloning repo...')
    !git clone {REPO_URL} {ROOT}
    os.chdir(ROOT)
else:
    os.chdir(ROOT)

print(f'  Working dir: {os.getcwd()}')

# Install dependencies
!pip install -q -e . 2>/dev/null || pip install -q -r requirements.txt 2>/dev/null || echo '  (deps already installed)'
!pip install -q huggingface_hub transformers bitsandbytes accelerate 2>/dev/null

print('  ✓ Dependencies ready')

## Cell 2: Setup Paths & Logger

In [ ]:
import sys
from pathlib import Path

# Determine ROOT
if Path('/kaggle/working/FLUX').exists():
    ROOT = Path('/kaggle/working/FLUX')
elif Path('/content/FLUX').exists():
    ROOT = Path('/content/FLUX')
else:
    ROOT = Path.cwd()
    if not (ROOT / 'flux_utils.py').exists():
        ROOT = Path('/Users/admin/Desktop/flux')

ROOT_DIR = ROOT
PHASES_DIR = ROOT / 'phases'
CHECKPOINT_DIR = ROOT / 'checkpoints'
CHECKPOINT_DIR.mkdir(exist_ok=True)

# Add paths
for p in [ROOT, PHASES_DIR / 'phase2', PHASES_DIR / 'phase_voice']:
    if p.exists() and str(p) not in sys.path:
        sys.path.insert(0, str(p))

from flux_utils import PhaseLogger, get_device, upload_flx_to_hf

log = PhaseLogger(phase=99)  # Phase Voice
log.separator("Phase Voice: Embed Qwen2.5-Omni into Flux-Apex")

print(f'  ROOT: {ROOT}')
print(f'  Checkpoints: {CHECKPOINT_DIR}')
print('  ✓ Paths configured')

## Cell 3: Hardware & Secrets

In [ ]:
import torch
import os

log.cell_start("Cell 3 — Hardware & Secrets")

device = get_device()
print(f'  Device: {device}')

if device == 'cuda':
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  GPU: {gpu_name} ({vram:.1f} GB)')
    
    # Check if enough VRAM for Qwen-Omni
    if vram < 15:
        print(f'  ⚠ Low VRAM ({vram:.1f} GB) — Qwen-Omni requires ~16GB for loading')
        print(f'    Will use aggressive quantization or CPU offload')
elif device == 'mps':
    print(f'  Using Apple Silicon MPS')
else:
    print(f'  ⚠ CPU only — this will be SLOW')

# Load HF token
hf_token = None

# Try Kaggle secrets
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret('HF_TOKEN')
    print('  ✓ Kaggle secrets loaded')
except:
    pass

# Try Colab secrets
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
        print('  ✓ Colab secrets loaded')
    except:
        pass

# Environment variable
if not hf_token:
    hf_token = os.environ.get('HF_TOKEN')
    if hf_token:
        print('  ✓ Environment HF_TOKEN loaded')

if hf_token:
    hf_token = hf_token.strip()
    os.environ['HF_TOKEN'] = hf_token
else:
    print('  ⚠ No HF_TOKEN found — some downloads may fail')

log.cell_end("Cell 3 — Hardware & Secrets", "PASS")

## Cell 4: Download Flux-Apex-V1.flx

In [ ]:
from huggingface_hub import hf_hub_download

log.cell_start("Cell 4 — Download Flux-Apex")

APEX_PATH = CHECKPOINT_DIR / 'Flux-Apex-V1.flx'

# Check local first
if APEX_PATH.exists():
    size_gb = APEX_PATH.stat().st_size / 1e9
    print(f'  ✓ Found local: {APEX_PATH.name} ({size_gb:.2f} GB)')
else:
    print(f'  Downloading Flux-Apex-V1.flx from HuggingFace...')
    try:
        downloaded = hf_hub_download(
            repo_id='UnseenGAP/FLUX',
            filename='checkpoints/Flux-Apex-V1.flx',
            local_dir=str(ROOT_DIR),
            token=hf_token,
        )
        size_gb = Path(downloaded).stat().st_size / 1e9
        print(f'  ✓ Downloaded: {size_gb:.2f} GB')
    except Exception as e:
        print(f'  ✗ Download failed: {e}')
        raise

# Verify
assert APEX_PATH.exists(), f"Flux-Apex not found at {APEX_PATH}"

log.metric("Flux-Apex size", f"{APEX_PATH.stat().st_size / 1e9:.2f} GB")
log.cell_end("Cell 4 — Download Flux-Apex", "PASS")

## Cell 5: Analyze Current Flux-Apex Structure

In [ ]:
import torch
from datetime import datetime

log.cell_start("Cell 5 — Analyze Flux-Apex")

print(f'\n  Loading {APEX_PATH.name} for analysis...')
apex = torch.load(str(APEX_PATH), map_location='cpu', weights_only=False)

print(f'\n  ═══ CURRENT FLUX-APEX STRUCTURE ═══')
print(f'  Format: {apex.get("format", "unknown")}')
print(f'  Version: {apex.get("version", "unknown")}')
print(f'  Phase: {apex.get("phase", "unknown")}')
print(f'  Top-level keys: {len(apex)}')

# Analyze components to remove
print(f'\n  Components to REMOVE:')

# Decoder
if 'decoder' in apex:
    decoder_size = 0
    if isinstance(apex['decoder'], dict):
        for k, v in apex['decoder'].items():
            if isinstance(v, dict):
                for kk, vv in v.items():
                    if isinstance(vv, torch.Tensor):
                        decoder_size += vv.numel() * vv.element_size()
            elif isinstance(v, torch.Tensor):
                decoder_size += v.numel() * v.element_size()
    print(f'    ✓ decoder: {decoder_size / 1e6:.1f} MB')
else:
    print(f'    ✗ decoder: not found')

# LLM reference
if 'llm' in apex:
    print(f'    ✓ llm: external reference to be removed')
if 'llm_reference' in apex:
    print(f'    ✓ llm_reference: external reference to be removed')

# Check for WaveToText (failed experiment)
wave_to_text_found = False
if 'adapters' in apex:
    adapters = apex['adapters']
    if isinstance(adapters, dict):
        for key in adapters.keys():
            if 'text' in key.lower() and 'wave' in key.lower():
                wave_to_text_found = True
                print(f'    ✓ adapters.{key}: failed experiment to remove')

if not wave_to_text_found:
    print(f'    ✓ WaveToText: not present (good - was failed experiment)')

# Components to keep
print(f'\n  Components to KEEP:')
keep_components = ['cse', 'field', 'memory', 'bridges', 'causal', 'adapters', 
                   'spatial_memory', 'causal_tracker', 'learned_rules', 'grid_to_wave']
for comp in keep_components:
    if comp in apex:
        print(f'    ✓ {comp}')
    else:
        print(f'    ✗ {comp}: not found')

log.cell_end("Cell 5 — Analyze Flux-Apex", "PASS")

## Cell 6: Download Qwen2.5-VL-3B-Instruct

This cell downloads the Qwen2.5-VL-3B model (~6GB fp16).

**Model specs:**
- `Qwen/Qwen2.5-VL-3B-Instruct`
- Text + Vision (images, video frames)
- ~3B parameters
- Native transformers support

**Requirements:**
- ~8GB disk space
- ~5-10 minutes on first run

In [ ]:
%%time
log.cell_start("Cell 6 — Download Qwen2.5-VL-3B")

MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'
VLM_PATH = CHECKPOINT_DIR / 'qwen_vl_3b.pt'

if VLM_PATH.exists():
    size_gb = VLM_PATH.stat().st_size / 1e9
    print(f'  ✓ Found existing: {VLM_PATH.name} ({size_gb:.2f} GB)')
    print(f'    Skipping download...')
    
    # Quick verify
    vlm_state = torch.load(str(VLM_PATH), map_location='cpu', weights_only=False)
    print(f'    Format: {vlm_state.get("format", "unknown")}')
    print(f'    Weights: {len(vlm_state.get("weights", {}))}')

else:
    print(f'  Downloading {MODEL_ID}...')
    print(f'  This will take 5-10 minutes.\n')
    
    try:
        from huggingface_hub import snapshot_download
        from safetensors import safe_open
        from pathlib import Path
        import json
        import gc
        
        # Step 1: Download model files
        print(f'  [1/4] Downloading from HuggingFace...')
        model_path = snapshot_download(
            MODEL_ID,
            token=hf_token,
            ignore_patterns=["*.bin", "*.md", "*.txt", "*.onnx"],
        )
        print(f'    Downloaded to: {model_path}')
        
        # Step 2: Load config
        print(f'  [2/4] Loading config...')
        config_path = Path(model_path) / 'config.json'
        with open(config_path, 'r') as f:
            config = json.load(f)
        
        print(f'    Architecture: {config.get("architectures", ["unknown"])[0]}')
        print(f'    Hidden size: {config.get("hidden_size", "unknown")}')
        print(f'    Vocab size: {config.get("vocab_size", "unknown")}')
        
        # Step 3: Load weights
        print(f'  [3/4] Loading safetensors...')
        
        vlm_state = {
            'format': 'FLUX_VLM',
            'version': '1.0',
            'base_model': MODEL_ID,
            'quantization': 'fp16',
            'timestamp': datetime.now().isoformat(),
            'config': config,
            'weights': {},
        }
        
        # Load tokenizer info
        tokenizer_path = Path(model_path) / 'tokenizer_config.json'
        if tokenizer_path.exists():
            with open(tokenizer_path, 'r') as f:
                vlm_state['tokenizer'] = json.load(f)
        
        # Find safetensor files
        safetensor_files = sorted(Path(model_path).glob('*.safetensors'))
        print(f'    Found {len(safetensor_files)} safetensor files')
        
        total_params = 0
        for st_file in safetensor_files:
            print(f'    Loading {st_file.name}...')
            with safe_open(st_file, framework="pt", device="cpu") as f:
                for key in f.keys():
                    tensor = f.get_tensor(key)
                    # Convert to fp16 to save space
                    if tensor.dtype == torch.float32:
                        tensor = tensor.half()
                    vlm_state['weights'][key] = tensor
                    total_params += tensor.numel()
            gc.collect()
        
        vlm_state['total_params'] = total_params
        print(f'    Total weights: {len(vlm_state["weights"])}')
        print(f'    Total params: {total_params:,}')
        
        # Step 4: Save
        print(f'  [4/4] Saving to {VLM_PATH}...')
        torch.save(vlm_state, str(VLM_PATH), pickle_protocol=4)
        
        size_gb = VLM_PATH.stat().st_size / 1e9
        print(f'  ✓ Saved: {VLM_PATH.name} ({size_gb:.2f} GB)')
        
        del vlm_state
        gc.collect()
        
    except ImportError as e:
        print(f'  ✗ Missing dependency: {e}')
        raise
    except Exception as e:
        print(f'  ✗ Download failed: {e}')
        import traceback
        traceback.print_exc()
        raise

log.metric("VLM size", f"{VLM_PATH.stat().st_size / 1e9:.2f} GB")
log.cell_end("Cell 6 — Download Qwen2.5-VL-3B", "PASS")

## Cell 7: Build New Flux-Apex with Embedded VLM

This is the core transformation:
1. Load Flux-Apex core components (cse, field, memory, bridges, etc.)
2. **Remove** decoder completely (not just legacy flag — DELETE it)
3. **Remove** llm and llm_reference  
4. **Add** vlm module (Qwen2.5-VL-3B weights)
5. Update version to 5.0-vlm-embedded

**The VLM replaces:**
- byte decoder (text generation)
- external LLM reference
- external VLM reference

**The VLM provides:**
- Text generation (no external downloads)
- Vision understanding (images, video frames)
- ~3B params embedded

In [ ]:
log.cell_start("Cell 7 — Build New Flux-Apex")

from datetime import datetime
import gc

VLM_PATH = CHECKPOINT_DIR / 'qwen_vl_3b.pt'
assert VLM_PATH.exists(), f"Missing: {VLM_PATH}"

# Load Flux-Apex
print(f'\n  Loading Flux-Apex...')
apex = torch.load(str(APEX_PATH), map_location='cpu', weights_only=False)
original_size = APEX_PATH.stat().st_size
print(f'  ✓ Flux-Apex loaded: {original_size / 1e9:.2f} GB')

# Load VLM
print(f'\n  Loading VLM...')
vlm_data = torch.load(str(VLM_PATH), map_location='cpu', weights_only=False)
print(f'  ✓ VLM loaded: {len(vlm_data.get("weights", {}))} weights')

# ─────────────────────────────────────────────
# STEP 1: Build new model structure
# ─────────────────────────────────────────────

print(f'\n  Building new Flux-Apex structure...')

new_apex = {
    # Header
    'format': 'FLUX',
    'version': '5.0-vlm-embedded',
    'phase': 'phase_vlm',
    'timestamp': datetime.now().isoformat(),
    'can_continue_learning': True,
}

# ─────────────────────────────────────────────
# STEP 2: Copy core FLUX components
# ─────────────────────────────────────────────

core_components = [
    'cse',           # Continuous Semantic Encoder
    'field',         # Resonance Field
    'memory',        # Three-tier memory
    'bridges',       # Wave ↔ Field projections
    'causal',        # CGN + causal graph
    'causal_tracker',
    'learned_rules',
    'spatial_memory',
    'grid_to_wave',
    'adapters',      # Grid, image, audio adapters
]

print(f'\n  Copying core components:')
for comp in core_components:
    if comp in apex:
        new_apex[comp] = apex[comp]
        print(f'    ✓ {comp}')
    else:
        print(f'    ✗ {comp}: not found')

# Copy runtime_config and other metadata
for key in ['runtime_config', 'state']:
    if key in apex:
        new_apex[key] = apex[key]

# ─────────────────────────────────────────────
# STEP 3: Track and skip removed components
# ─────────────────────────────────────────────

print(f'\n  Components being REMOVED:')
removed_components = []
for comp in ['decoder', 'llm', 'llm_reference', 'grid_adapters']:
    if comp in apex:
        print(f'    ✓ {comp}: REMOVED')
        removed_components.append(comp)

# Free apex memory
del apex
gc.collect()

# ─────────────────────────────────────────────
# STEP 4: Add VLM module
# ─────────────────────────────────────────────

print(f'\n  Adding VLM module (Qwen2.5-VL-3B)...')

new_apex['vlm'] = {
    'format_version': '1.0',
    'base_model': vlm_data.get('base_model', 'Qwen/Qwen2.5-VL-3B-Instruct'),
    'quantization': vlm_data.get('quantization', 'fp16'),
    'timestamp': datetime.now().isoformat(),
    'total_params': vlm_data.get('total_params', 0),
    
    'config': vlm_data.get('config', {}),
    'tokenizer': vlm_data.get('tokenizer', {}),
    'weights': vlm_data.get('weights', {}),
    
    # Bridge config for FLUX integration
    'bridges': {
        'wave_to_vlm': {'wave_dim': 432, 'vlm_dim': 2048},  # Qwen2.5-3B hidden_size
        'vlm_to_wave': {'vlm_dim': 2048, 'wave_dim': 432},
    },
}

print(f'    ✓ Weights: {len(new_apex["vlm"]["weights"])}')
print(f'    ✓ Params: {new_apex["vlm"]["total_params"]:,}')

# Free vlm_data
del vlm_data
gc.collect()

# ─────────────────────────────────────────────
# STEP 5: Update components flags
# ─────────────────────────────────────────────

print(f'\n  Updating component flags...')

new_apex['components'] = {
    # Perception
    'cse': True,
    'grid_to_wave': True,
    'spatial_memory': True,
    'perception_field': False,
    
    # Knowledge
    'field': True,
    'working_memory': True,
    'episodic_memory': True,
    'semantic_memory': True,
    
    # Generation — NEW: VLM replaces decoder/llm
    'vlm': True,
    'vlm_text': True,
    'vlm_vision': True,
    
    # Generation — REMOVED
    'decoder': False,
    'llm': False,
    
    # Reasoning
    'causal_tracker': True,
    'rule_inducer': False,
    'goal_planner': False,
    'causal_graph': True,
    
    # Bridges & Adapters
    'bridges': True,
    'adapters': True,
    'learned_rules': True,
}

# ─────────────────────────────────────────────
# STEP 6: Update runtime_config
# ─────────────────────────────────────────────

print(f'  Updating runtime config...')

if 'runtime_config' not in new_apex:
    new_apex['runtime_config'] = {}

new_apex['runtime_config']['generation'] = {
    'vlm_primary': True,
    'llm_primary': False,
    'byte_decoder_enabled': False,
    'generation_mode': 'vlm',
}

new_apex['runtime_config']['vlm'] = {
    'enabled': True,
    'model_type': 'qwen2_vl',
    'model_name': 'Qwen/Qwen2.5-VL-3B-Instruct',
    'quantization': 'fp16',
    'hidden_dim': 2048,
    'max_tokens': 2048,
    'temperature': 0.7,
    'text_enabled': True,
    'vision_enabled': True,
    'use_flux_context': True,
    'flux_context_limit': 10,
}

# ─────────────────────────────────────────────
# STEP 7: Update metadata
# ─────────────────────────────────────────────

print(f'  Updating metadata...')

new_apex['metadata'] = {
    'last_modified': datetime.now().isoformat(),
    'modified_components': ['vlm'],
    'removed_components': removed_components,
    'vlm_embedded': True,
    'vlm_base_model': 'Qwen/Qwen2.5-VL-3B-Instruct',
    'vlm_quantization': 'fp16',
    'no_external_downloads': True,
    'capabilities': ['text', 'grid', 'image', 'audio', 'vision'],
    'phase': 'vlm',
    'description': 'FLUX with embedded Qwen2.5-VL-3B — self-contained text + vision AI',
}

new_apex['modified'] = True
new_apex['modified_components'] = ['vlm']

print(f'\n  ✓ New Flux-Apex structure built')
print(f'    Version: {new_apex["version"]}')
print(f'    VLM params: {new_apex["vlm"]["total_params"]:,}')

log.cell_end("Cell 7 — Build New Flux-Apex", "PASS")

## Cell 8: Verify New Structure

In [ ]:
log.cell_start("Cell 8 — Verify Structure")

print(f'\n  ═══ NEW FLUX-APEX STRUCTURE ═══')
print(f'  Version: {new_apex["version"]}')
print(f'  Phase: {new_apex["phase"]}')

print(f'\n  Top-level keys ({len(new_apex)}):')
for key in sorted(new_apex.keys()):
    if key == 'vlm':
        print(f'    ✓ {key} (NEW - embedded Qwen2.5-VL-3B)')
    elif key in ['decoder', 'llm', 'llm_reference', 'grid_adapters']:
        print(f'    ✗ {key} (REMOVED)')
    else:
        print(f'    • {key}')

# Verify critical components
print(f'\n  Critical components check:')
critical = ['cse', 'field', 'memory', 'bridges', 'vlm']
all_present = True
for comp in critical:
    if comp in new_apex:
        print(f'    ✓ {comp}: present')
    else:
        print(f'    ✗ {comp}: MISSING!')
        all_present = False

# Verify VLM structure
print(f'\n  VLM module check:')
if 'vlm' in new_apex:
    vlm = new_apex['vlm']
    print(f'    ✓ base_model: {vlm.get("base_model")}')
    print(f'    ✓ weights: {len(vlm.get("weights", {}))}')
    print(f'    ✓ total_params: {vlm.get("total_params", 0):,}')
else:
    print(f'    ✗ VLM module not found!')
    all_present = False

# Verify removed components
print(f'\n  Removed components check:')
removed = ['decoder', 'llm', 'llm_reference', 'grid_adapters']
for comp in removed:
    if comp in new_apex:
        print(f'    ✗ {comp}: should be removed!')
        all_present = False
    else:
        print(f'    ✓ {comp}: removed')

if all_present:
    print(f'\n  ✓ Structure verification PASSED')
else:
    print(f'\n  ✗ Structure verification FAILED')

log.cell_end("Cell 8 — Verify Structure", "PASS" if all_present else "FAIL")

## Cell 9: Save New Flux-Apex-V1.flx

In [ ]:
log.cell_start("Cell 9 — Save Flux-Apex")

import shutil
import gc

# Check available disk space
total, used, free = shutil.disk_usage(CHECKPOINT_DIR)
print(f'  Disk space: {free / 1e9:.1f} GB free / {total / 1e9:.1f} GB total')

# Estimated size: ~5.8GB original - 0.1GB decoder + ~6GB VLM = ~11.7GB
estimated_size_gb = 12

if free / 1e9 < estimated_size_gb + 2:
    print(f'  ⚠ Low disk space! Need ~{estimated_size_gb + 2} GB, have {free / 1e9:.1f} GB')
    
    # Delete VLM file since it's now embedded
    if VLM_PATH.exists():
        VLM_PATH.unlink()
        print(f'  Deleted: {VLM_PATH.name} to free space')
    gc.collect()
    
    total, used, free = shutil.disk_usage(CHECKPOINT_DIR)
    print(f'  Disk space now: {free / 1e9:.1f} GB free')

# Backup original (if space allows)
BACKUP_PATH = CHECKPOINT_DIR / 'Flux-Apex-V1-backup-v4.flx'
if not BACKUP_PATH.exists() and free / 1e9 > estimated_size_gb + 8:
    print(f'\n  Creating backup: {BACKUP_PATH.name}')
    shutil.copy(APEX_PATH, BACKUP_PATH)
    print(f'  ✓ Backup created')
elif BACKUP_PATH.exists():
    print(f'\n  Backup already exists: {BACKUP_PATH.name}')
else:
    print(f'\n  ⚠ Skipping backup (low disk space)')

# Save new model
print(f'\n  Saving new Flux-Apex-V1.flx...')
print(f'  (Estimated size: ~{estimated_size_gb} GB)\n')

torch.save(new_apex, str(APEX_PATH), pickle_protocol=4)

new_size = APEX_PATH.stat().st_size
size_change = new_size - original_size

print(f'  ✓ Saved: {APEX_PATH.name}')
print(f'\n  Size comparison:')
print(f'    Original: {original_size / 1e9:.2f} GB')
print(f'    New:      {new_size / 1e9:.2f} GB')
print(f'    Change:   {size_change / 1e9:+.2f} GB ({size_change / original_size * 100:+.1f}%)')

# Clean up VLM file if still exists
if VLM_PATH.exists():
    VLM_PATH.unlink()
    print(f'\n  Cleaned up: {VLM_PATH.name}')

gc.collect()

log.metric("New file size", f"{new_size / 1e9:.2f} GB")
log.metric("Size change", f"{size_change / 1e9:+.2f} GB")

log.cell_end("Cell 9 — Save Flux-Apex", "PASS")

## Cell 10: Verify Saved Model

In [ ]:
log.cell_start("Cell 10 — Verify Saved Model")

import gc

print(f'\n  Reloading saved model for verification...')
print(f'  (Loading ~{APEX_PATH.stat().st_size / 1e9:.1f} GB file...)\n')

# Clear memory first
del new_apex
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# Reload
verified = torch.load(str(APEX_PATH), map_location='cpu', weights_only=False)

print(f'  ═══ VERIFIED FLUX-APEX ═══')
print(f'  Format: {verified.get("format")}')
print(f'  Version: {verified.get("version")}')
print(f'  Phase: {verified.get("phase")}')

# Check VLM module
print(f'\n  VLM module:')
if 'vlm' in verified:
    vlm = verified['vlm']
    print(f'    ✓ Base model: {vlm.get("base_model")}')
    print(f'    ✓ Quantization: {vlm.get("quantization")}')
    print(f'    ✓ Weights: {len(vlm.get("weights", {}))}')
    print(f'    ✓ Total params: {vlm.get("total_params", 0):,}')
    vlm_ok = True
else:
    print(f'    ✗ VLM module not found!')
    vlm_ok = False

# Check removed components
print(f'\n  Removed components:')
removed_ok = True
for comp in ['decoder', 'llm', 'llm_reference', 'grid_adapters']:
    if comp in verified:
        print(f'    ✗ {comp}: still present (ERROR)')
        removed_ok = False
    else:
        print(f'    ✓ {comp}: removed')

# Check core components
print(f'\n  Core components:')
core_ok = True
for comp in ['cse', 'field', 'memory', 'bridges']:
    if comp in verified:
        print(f'    ✓ {comp}: present')
    else:
        print(f'    ✗ {comp}: MISSING')
        core_ok = False

# Check component flags
print(f'\n  Component flags:')
comps = verified.get('components', {})
print(f'    vlm: {comps.get("vlm", False)}')
print(f'    vlm_text: {comps.get("vlm_text", False)}')
print(f'    vlm_vision: {comps.get("vlm_vision", False)}')
print(f'    decoder: {comps.get("decoder", False)}')
print(f'    llm: {comps.get("llm", False)}')

# Final verdict
verification_passed = (
    verified.get('format') == 'FLUX' and
    verified.get('version') == '5.0-vlm-embedded' and
    vlm_ok and removed_ok and core_ok
)

if verification_passed:
    print(f'\n  ═══════════════════════════════════════')
    print(f'  ═══ VERIFICATION PASSED ═══')
    print(f'  ═══════════════════════════════════════')
    log.success("Model verification passed!")
else:
    print(f'\n  ═══ VERIFICATION FAILED ═══')
    log.error("Model verification failed!")

log.cell_end("Cell 10 — Verify Saved Model", "PASS" if verification_passed else "FAIL")

## Cell 11: Upload to HuggingFace Hub

In [ ]:
log.cell_start("Cell 11 — Upload to HuggingFace")

if hf_token:
    print(f'\n  Uploading to HuggingFace Hub...')
    print(f'  File: {APEX_PATH.name} ({APEX_PATH.stat().st_size / 1e9:.2f} GB)')
    print(f'  This may take several minutes...\n')
    
    try:
        success = upload_flx_to_hf(
            str(APEX_PATH),
            hf_token=hf_token,
        )
        
        if success:
            log.success("Uploaded to HuggingFace Hub!")
            print(f'\n  ✓ View at: https://huggingface.co/UnseenGAP/FLUX')
        else:
            log.warning("Upload failed")
    except Exception as e:
        print(f'  ✗ Upload error: {e}')
        log.warning(f"Upload error: {str(e)[:50]}")
else:
    print(f'  ⚠ No HF_TOKEN — skipping upload')
    print(f'  Model saved locally at: {APEX_PATH}')

log.cell_end("Cell 11 — Upload to HuggingFace", "PASS")

## Cell 12: Update Documentation

In [ ]:
log.cell_start("Cell 12 — Update Documentation")

# Update FLUX_APEX_V1.md
docs_path = ROOT / 'DOCS' / 'FLUX_APEX_V1.md'

if docs_path.exists():
    print(f'  Checking {docs_path.name}...')
    
    content = docs_path.read_text()
    
    if '5.0-vlm-embedded' in content:
        print(f'  ✓ Already updated for v5.0-vlm-embedded')
    else:
        print(f'  ⚠ Documentation needs manual update to v5.0')
        print(f'    Key changes:')
        print(f'    - Version: 4.0 → 5.0-vlm-embedded')
        print(f'    - Added: vlm component (Qwen2.5-VL-3B-Instruct)')
        print(f'    - Removed: decoder, llm, llm_reference')
        print(f'    - No external downloads required')
else:
    print(f'  ⚠ Documentation file not found: {docs_path}')

log.cell_end("Cell 12 — Update Documentation", "PASS")

## Cell 13: Summary

In [ ]:
log.separator("Phase VLM Complete")

original_gb = original_size / 1e9
new_gb = APEX_PATH.stat().st_size / 1e9
change_gb = new_gb - original_gb

print(f"""
╔════════════════════════════════════════════════════════════════════╗
║                     PHASE VLM COMPLETE                             ║
║              FLUX IS NOW SELF-CONTAINED                            ║
╠════════════════════════════════════════════════════════════════════╣
║                                                                    ║
║  WHAT CHANGED:                                                     ║
║  ├── Version: 4.0 → 5.0-vlm-embedded                              ║
║  ├── ADDED: vlm module (Qwen2.5-VL-3B-Instruct, fp16)             ║
║  │   ├── Text generation (~3B params)                             ║
║  │   └── Vision understanding (images, video frames)              ║
║  └── REMOVED completely:                                          ║
║      ├── decoder (byte-level, replaced by VLM)                    ║
║      ├── llm (external reference)                                 ║
║      ├── llm_reference (external reference)                       ║
║      └── grid_adapters (legacy duplicate)                         ║
║                                                                    ║
║  SIZE:                                                             ║
║  ├── Original: {original_gb:>6.2f} GB                                       ║
║  ├── New:      {new_gb:>6.2f} GB                                       ║
║  └── Change:   {change_gb:>+6.2f} GB                                        ║
║                                                                    ║
║  CAPABILITIES:                                                     ║
║  ├── ✓ Text generation (no external LLM needed)                   ║
║  ├── ✓ Vision understanding (images + video)                      ║
║  ├── ✓ Grid reasoning (ARC-AGI)                                   ║
║  ├── ✓ Wave-based memory + field                                  ║
║  └── ✓ No runtime downloads required                              ║
║                                                                    ║
║  OUTPUT:                                                           ║
║  └── {str(APEX_PATH):56} ║
║                                                                    ║
╚════════════════════════════════════════════════════════════════════╝
""")

print(f"The model is now FULLY SELF-CONTAINED.")
print(f"No external downloads. Embedded text + vision capabilities.")

log.success("Phase VLM complete — FLUX is now self-contained!")

---

## Optional: Test VLM Generation

This requires setting up the VLM for inference with the embedded weights.

In [ ]:
# Optional: Quick inspection of embedded VLM
print(f'\n  Inspecting embedded VLM...')

# Load the saved model
apex = torch.load(str(APEX_PATH), map_location='cpu', weights_only=False)

if 'vlm' in apex:
    vlm = apex['vlm']
    print(f'\n  ═══ EMBEDDED VLM ═══')
    print(f'  Base model: {vlm.get("base_model")}')
    print(f'  Quantization: {vlm.get("quantization")}')
    print(f'  Total params: {vlm.get("total_params", 0):,}')
    
    # Show some weight keys
    weights = vlm.get('weights', {})
    print(f'\n  Sample weight keys ({len(weights)} total):')
    for i, key in enumerate(list(weights.keys())[:10]):
        tensor = weights[key]
        print(f'    {key}: {tensor.shape} {tensor.dtype}')
    if len(weights) > 10:
        print(f'    ... and {len(weights) - 10} more')
    
    # Show config
    config = vlm.get('config', {})
    print(f'\n  Config:')
    print(f'    hidden_size: {config.get("hidden_size", "unknown")}')
    print(f'    num_hidden_layers: {config.get("num_hidden_layers", "unknown")}')
    print(f'    vocab_size: {config.get("vocab_size", "unknown")}')
    
    # Test bridge dimensions
    print(f'\n  Bridge config:')
    bridges = vlm.get('bridges', {})
    print(f'    wave_to_vlm: {bridges.get("wave_to_vlm", {})}')
    print(f'    vlm_to_wave: {bridges.get("vlm_to_wave", {})}')
    
    print(f'\n  ✓ VLM inspection complete')
else:
    print(f'  ✗ VLM not found in model')

del apex